> **What you already know:** PyTorch `nn.Module`, `DataLoader`, four-step training loop, `state_dict` save/load (from `00-pytorch-primer`).
> **The gap this chapter closes:** You cannot yet understand how autograd builds a computation graph, or write a network that processes a sequence of steps.
> **What you'll have by the end:** A trained `HousePriceModel` with autograd-verified gradients and a path to pretrained music generation.
> **What this chapter is not:** A production RNN implementation — that is `01-rnns/` Part 2.


<table align="center">
  <td align="center"><a target="_blank" href="http://introtodeeplearning.com">
        <img src="https://i.ibb.co/Jr88sn2/mit.png" style="padding-bottom:5px;" />
      Visit MIT Deep Learning</a></td>
  <td align="center"><a target="_blank" href="https://colab.research.google.com/github/MITDeepLearning/introtodeeplearning/blob/master/lab1/PT_Part1_Intro.ipynb">
        <img src="https://i.ibb.co/2P3SLwK/colab.png"  style="padding-bottom:5px;" />Run in Google Colab</a></td>
  <td align="center"><a target="_blank" href="https://github.com/MITDeepLearning/introtodeeplearning/blob/master/lab1/PT_Part1_Intro.ipynb">
        <img src="https://i.ibb.co/xfJbPmL/github.png"  height="70px" style="padding-bottom:5px;"  />View Source on GitHub</a></td>
</table>

# Copyright Information


In [ ]:
# Copyright 2026 MIT Introduction to Deep Learning. All Rights Reserved.
#
# Licensed under the MIT License. You may not use this file except in compliance
# with the License. Use and/or modification of this code outside of MIT Introduction
# to Deep Learning must reference:
#
# © MIT Introduction to Deep Learning
# http://introtodeeplearning.com
#

# PyTorch from First Principles

## Building Deep Learning Intuition One Tensor at a Time

This notebook builds the **complete mental model for PyTorch's tensor abstraction and autograd system** — starting with a running example that threads through every concept.

Every concept is demonstrated on the same problem:

> **Predicting house prices from size and age**  
> 5 houses, 2 features (sq ft, age) → 1 price. Small enough to visualise, real enough to matter.

| Part | Concept                    | Key Idea                                                                           |
| ---- | -------------------------- | ---------------------------------------------------------------------------------- |
| 1    | Tensors as Data Containers | Scalars → vectors → matrices → batches; faster + safer than lists                  |
| 2    | Computations on Tensors    | PyTorch traces operations into a graph; hand-tuning weights fails at scale         |
| 3    | Neural Networks in PyTorch | `nn.Module` wraps learnable `nn.Parameter`s; `forward()` defines the computation   |
| 4    | Automatic Differentiation  | `.backward()` computes ∂loss/∂every_weight in one call; gradient descent converges |
| 5    | From Toy to Production     | Same autograd scales from 3 params to 1.5 billion (GPT-2)                          |
| 6    | Music Generation Bonus     | Pre-trained transformers (MusicGen, TunesFormer) show inference at scale           |

---

## 0. Setup

[PyTorch](https://pytorch.org/) is a deep learning library known for flexibility and ease of use. For all labs in Introduction to Deep Learning 2026, a PyTorch version is available.


---

## Prerequisite Bridge — From `00-pytorch-primer`

| Foundation (from `00-pytorch-primer/keras-to-pytorch-primer.ipynb`) | Role in this notebook                                                                                      |
| ------------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------- |
| `nn.Module` subclassing, `__init__` + `forward()`                   | Every model here — `OurDenseLayer`, `HousePriceModel` — is an `nn.Module`; the pattern is not re-explained |
| The 4-step training loop (`zero_grad → forward → backward → step`)  | The exact loop pattern used in Parts 4–5; already proved correct in the primer                             |
| `CrossEntropyLoss` fuses `log_softmax + nll_loss`                   | Used in Parts 4–5; the proof from the primer is not repeated                                               |
| `model.train()` / `model.eval()` + `torch.no_grad()`                | Standard wrapping for training and inference cells throughout                                              |
| `state_dict()` save/load round-trip                                 | Carries through to the music generation section where pretrained weights are loaded                        |

> **If you haven't completed `00-pytorch-primer/keras-to-pytorch-primer.ipynb`** the `nn.Module` and training-loop patterns used here will feel unfamiliar. That notebook takes 30–40 minutes and pays off immediately here.


### First-Principles Roadmap

![Deep learning first-principles roadmap from tensors through automatic differentiation, training, scaling, and pretrained music generation](images/deep-learning-first-principles-roadmap.png)

This notebook builds one learning cycle from first principles: data becomes tensor computations, a model produces predictions, a loss produces gradients, and an optimizer updates parameters. The later production and music-generation sections reuse the same core ideas at a different scale.


## Table of Contents

1. [Where This Notebook Sits in the Larger Topic Space](#where-this-notebook-sits-in-the-larger-topic-space)
2. [Setup](#0-setup)
3. [Part 1 — Tensors as Data Containers](#part-1--tensors-as-data-containers)
   - [Code Walkthrough: Three Reasons Tensors Beat Python Lists, Recapped](#code-walkthrough-three-reasons-tensors-beat-python-lists-recapped)
4. [Part 2 — Computations on Tensors](#part-2--computations-on-tensors)
5. [Part 3 — Neural Networks in PyTorch](#part-3--neural-networks-in-pytorch)
6. [Part 4 — Automatic Differentiation](#part-4--automatic-differentiation)
   - [Code Walkthrough: Manual Gradient Descent on a Parabola](#code-walkthrough-manual-gradient-descent-on-a-parabola)
   - [Training Loop to Convergence](#training-loop-to-convergence)
7. [Part 5 — From Toy to Production: Same Machinery, Bigger Numbers](#part-5--from-toy-to-production-same-machinery-bigger-numbers)
8. [Part 6 — Music Generation with HuggingFace](#part-6--music-generation-with-huggingface)
   - [Why Pre-trained Models Instead of Training From Scratch?](#why-pre-trained-models-instead-of-training-from-scratch)
   - [Option A — MusicGen (text-prompt → audio)](#option-a--musicgen-text-prompt--audio)
   - [Option B — TunesFormer (ABC seed → ABC notation)](#option-b--tunesformer-abc-seed--abc-notation)
9. [What This Notebook Covered (and What It Didn't)](#what-this-notebook-covered-and-what-it-didnt)
10. [Summary — What You Built](#summary--what-you-built)

> Links jump to the matching heading below. If a link doesn't scroll correctly in your Jupyter
> viewer, use `Ctrl+F` / the notebook outline panel with the section title instead.


### Where This Notebook Sits in the Larger Topic Space

"PyTorch fundamentals" is a wide subject — tensor mechanics, autograd internals, model-building APIs,
training-loop mechanics, and production-scale deployment could each fill their own notebook. This
notebook is deliberately scoped to the mental model that makes everything else fall out: tensors →
computation graphs → `nn.Module` → autograd → training loop → same machinery at scale.

| Topic area        | Covered here                                                                         | Deliberately out of scope                                                            |
| ----------------- | ------------------------------------------------------------------------------------ | ------------------------------------------------------------------------------------ |
| Tensor mechanics  | Creation, shape/`ndim`, indexing, NumPy interop                                      | Reshaping (`view`/`permute`), broadcasting rules, dtype casting, in-place ops        |
| Autograd          | `.backward()`, `requires_grad`, gradient inspection                                  | Custom `autograd.Function`, higher-order gradients, vector-Jacobian products         |
| Model building    | `nn.Parameter`, `nn.Sequential`, subclassing + control flow                          | `nn.Conv2d`/`nn.LSTM`/`nn.Embedding` (sequence/vision-specific layers)               |
| Training loop     | Full loop to convergence with a real loss curve                                      | Mini-batching (`DataLoader`), train/val split, regularization, LR scheduling         |
| Production scale  | Real parameter counts (ResNet-18), real pretrained inference (MusicGen, TunesFormer) | Fine-tuning a pretrained model, mixed precision, deployment/export                   |
| Sequence modeling | —                                                                                    | RNN/LSTM/GRU mechanics, attention — this is **Lab 1**; sequence models are **Lab 2** |

The complete breakdown — everything implemented, explained-but-not-built, or named-and-skipped — is
in the closing **"What This Notebook Covered (and What It Didn't)"** section.


In [ ]:
import torch
import torch.nn as nn

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# ── Deterministic seeds for reproducible results ──────────────────────────────
torch.manual_seed(42)
np.random.seed(42)
print("Seeds set → every run produces identical results.")

### Predict before you run

Before running the upcoming scatter plot, commit to an answer: which feature will show the stronger visual correlation with house price?

1. **Size** — bigger houses always cost more; age barely matters.
2. **Age** — newer houses cost more; size is secondary.
3. **Both roughly equally** — the plot will show two similarly strong linear trends.


In [ ]:
# ── Our Running Example — House Price Prediction ──────────────────────────────
# Throughout this notebook, every concept is demonstrated on the same 5 houses.
houses = torch.tensor(
    [
        [1200.0, 10.0],  # size (sq ft), age (years)
        [1500.0, 5.0],
        [800.0, 15.0],
        [2000.0, 2.0],
        [1000.0, 12.0],
    ],
    dtype=torch.float32,
)
prices = torch.tensor([250.0, 320.0, 180.0, 450.0, 210.0])  # $1000s

print("Our running example — 5 houses:")
print("  Size (sq ft)  Age (yrs)  Price ($k)")
for i, (h, p) in enumerate(zip(houses, prices)):
    print(f"  [{i}]  {h[0]:6.0f}      {h[1]:4.0f}       {p:6.0f}")

With the 5 houses defined, let's visualise how each feature relates to price — one subplot per
feature, colored by the _other_ feature so both dimensions are visible at once.


In [ ]:
# ── 2-panel scatter plot of our running example ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sc0 = axes[0].scatter(
    houses[:, 0], prices, c=houses[:, 1], cmap="viridis", s=100, edgecolor="black"
)
axes[0].set_xlabel("Size (sq ft)")
axes[0].set_ylabel("Price ($1000s)")
axes[0].set_title("Price vs Size (color = age)")
plt.colorbar(sc0, ax=axes[0], label="Age (years)")
sc1 = axes[1].scatter(
    houses[:, 1], prices, c=houses[:, 0], cmap="plasma", s=100, edgecolor="black"
)
axes[1].set_xlabel("Age (years)")
axes[1].set_ylabel("Price ($1000s)")
axes[1].set_title("Price vs Age (color = size)")
plt.colorbar(sc1, ax=axes[1], label="Size (sq ft)")
plt.suptitle("Our Running Example: 5 Houses → 1 Price", fontweight="bold")
plt.tight_layout()
plt.show()
print(
    "This dataset is our 'the cat sat on the mat' — every concept demonstrated on these 5 houses."
)

#### What just happened — and what's missing

We established the running example: 5 houses, 2 input features (size, age), 1 price label. The scatter plots confirm that size correlates positively with price while age correlates negatively — but neither feature alone fully determines the price.

What's missing: we have raw Python data but no structure that can batch, broadcast, or GPU-accelerate it. For that we need **tensors** — the topic of Part 1.


---

## Part 1 — Tensors as Data Containers

PyTorch is a machine learning library, like TensorFlow. At its core, PyTorch provides an interface for creating and manipulating [tensors](https://pytorch.org/docs/stable/tensors.html), which are data structures that you can think of as multi-dimensional arrays. Tensors are represented as n-dimensional arrays of base datatypes such as a string or integer -- they provide a way to generalize vectors and matrices to higher dimensions. PyTorch provides the ability to perform computation on these tensors, define neural networks, and train them efficiently.

The [`shape`](https://pytorch.org/docs/stable/generated/torch.Tensor.shape.html#torch.Tensor.shape) of a PyTorch tensor defines its number of dimensions and the size of each dimension. The `ndim` or [`dim`](https://pytorch.org/docs/stable/generated/torch.Tensor.dim.html#torch.Tensor.dim) of a PyTorch tensor provides the number of dimensions (n-dimensions) -- this is equivalent to the tensor's rank (as is used in TensorFlow), and you can also think of this as the tensor's order or degree.

Let's start by creating some tensors and inspecting their properties:


#### **Predict first** — why do we need tensors?

Python has lists. NumPy has arrays. Predict which of these are true about `torch.Tensor` vs nested lists:

1. **Speed**: Can tensors outrun nested lists on matrix multiply?
2. **Shape safety**: Will tensors catch a shape mismatch that lists silently swallow?
3. **GPU support**: Can a list of lists run on a GPU?

Make your prediction, then run the next three cells — one measured proof per reason.


In [ ]:
# ── Reason 1: Speed — matrix multiply ─────────────────────────────────────────
import time

n = 1000
np_a = np.random.randn(n, n).astype(np.float32)
np_b = np.random.randn(n, n).astype(np.float32)
t0 = time.time()
_ = np_a @ np_b
np_time = time.time() - t0

t_a = torch.from_numpy(np_a)
t_b = torch.from_numpy(np_b)
t0 = time.time()
_ = torch.matmul(t_a, t_b)
torch_time = time.time() - t0

print(f"Matrix multiply (1000×1000):")
print(f"  NumPy:  {np_time*1000:.1f} ms")
print(
    f"  PyTorch: {torch_time*1000:.1f} ms  ({np_time/max(torch_time,1e-9):.1f}× speedup)"
)

**Reason 2 — shape safety.** Nested Python lists don't check that a "matrix" has a consistent
shape at all; a bug that mismatches dimensions can silently propagate for many lines before it
surfaces. Does `torch.Tensor` catch a shape mismatch immediately, at the point of the mistake?


In [ ]:
# ── Reason 2: Shape safety ─────────────────────────────────────────────────────
try:
    bad = torch.randn(3, 4) + torch.randn(5, 6)
except RuntimeError as e:
    print(f"✓ Shape mismatch caught: '{e}'")
    print("  Lists would silently fail or give wrong results.")

**Reason 3 — GPU support.** Deep learning workloads are only practical at scale because they run
on GPUs. Can a plain Python list of lists move to a GPU the way a tensor can?


In [ ]:
# ── Reason 3: GPU support ──────────────────────────────────────────────────────
print(f"GPU available: {torch.cuda.is_available()}")
print("  → torch.Tensor can move to GPU with .to('cuda'); lists cannot.")
print("\n→ Tensors are PURPOSE-BUILT for deep learning: fast, safe, GPU-ready.")

### Code Walkthrough: Three Reasons Tensors Beat Python Lists, Recapped

**What just ran — 3 key patterns, one per cell above:**

---

**`torch.from_numpy(np_a)` — zero-copy bridge from NumPy**
Converts a NumPy array to a PyTorch tensor sharing the same memory block — no copy overhead. This benchmarks NumPy vs. PyTorch matmul on **identical** data so the speed comparison is fair. The speedup emerges because PyTorch dispatches to optimized BLAS kernels; Python lists use interpreted loops with per-element object overhead.

---

**`torch.randn(3, 4) + torch.randn(5, 6)` → `RuntimeError`**
PyTorch validates shapes at every operation and raises immediately. Python lists would either silently truncate or produce a confusing `IndexError` far away from the actual mistake — making bugs much harder to trace. The intentional `RuntimeError` here demonstrates **fail-safe behavior**.

---

**`torch.cuda.is_available()`**
Returns `True` if a CUDA GPU is accessible. Calling `.to('cuda')` on any tensor then moves it to GPU memory and all subsequent math on that tensor runs on the GPU. Python lists have no `.to()` equivalent — GPU acceleration requires the tensor abstraction.

> **Key takeaway:** Tensors are purpose-built for deep learning — fast (BLAS-dispatched), safe (shape-checked at every op), and GPU-portable. Every higher-level PyTorch construct — `nn.Module`, `autograd`, `DataLoader` — is built on this foundation.


In [ ]:
# ── Scalar tensors (0-D) ──────────────────────────────────────────────────────
integer = torch.tensor(1234)
decimal = torch.tensor(3.14159265359)

print(f"`integer` is a {integer.ndim}-d Tensor: {integer}")
print(f"`decimal` is a {decimal.ndim}-d Tensor: {decimal}")

Vectors and lists can be used to create 1-d tensors:


In [ ]:
# ── 1-D Tensors (vectors) ─────────────────────────────────────────────────────
fibonacci = torch.tensor([1, 1, 2, 3, 5, 8])
count_to_100 = torch.tensor(range(100))

print(f"`fibonacci` is a {fibonacci.ndim}-d Tensor with shape: {fibonacci.shape}")
print(
    f"`count_to_100` is a {count_to_100.ndim}-d Tensor with shape: {count_to_100.shape}"
)

Next, let’s create 2-d (i.e., matrices) and higher-rank tensors. In image processing and computer vision, we will use 4-d Tensors with dimensions corresponding to batch size, number of color channels, image height, and image width.


In [ ]:
# ── 2-D and higher-rank Tensors ───────────────────────────────────────────────

"""TODO: Define a 2-d Tensor"""

matrix = torch.tensor([[1, 2, 3], [4, 5, 6]])

assert isinstance(matrix, torch.Tensor), "matrix must be a torch Tensor object"
assert matrix.ndim == 2

"""TODO: Define a 4-d Tensor."""
# Use torch.zeros to initialize a 4-d Tensor of zeros with size 10 x 3 x 256 x 256.
#   You can think of this as 10 images where each image is RGB 256 x 256.
images = torch.zeros(10, 3, 256, 256)

assert isinstance(images, torch.Tensor), "images must be a torch Tensor object"
assert images.ndim == 4, "images must have 4 dimensions"
assert images.shape == (10, 3, 256, 256), "images is incorrect shape"
print(f"images is a {images.ndim}-d Tensor with shape: {images.shape}")

As you have seen, the `shape` of a tensor provides the number of elements in each tensor dimension. The `shape` is quite useful, and we'll use it often. You can also use slicing to access subtensors within a higher-rank tensor:


In [ ]:
# ── Tensor slicing — accessing sub-tensors ────────────────────────────────────
row_vector = matrix[1]
column_vector = matrix[:, 1]
scalar = matrix[0, 1]

print(f"`row_vector`: {row_vector}")
print(f"`column_vector`: {column_vector}")
print(f"`scalar`: {scalar}")

#### What just happened — and what's missing

Tensors are PyTorch's data container: fast (GPU-ready), safe (shape-checked), and built for batches. Every model input (images, text, audio) becomes a tensor before processing.

**Missing piece**: We have the _container_, but we haven't built anything that _learns_ yet. For that, we need operations that PyTorch can differentiate — next.


---

## Part 2 — Computations on Tensors

A convenient way to think about and visualize computations in a machine learning framework like PyTorch is in terms of graphs. We can define this graph in terms of tensors, which hold data, and the mathematical operations that act on these tensors in some order. Let's look at a simple example, and define this computation using PyTorch:

![alt text](images/external/add-graph.png)


In [ ]:
# ── Simple computation graph — node addition ──────────────────────────────────
# Create the nodes in the graph and initialize values
a = torch.tensor(15)
b = torch.tensor(61)

# Add them!
c1 = torch.add(a, b)
c2 = a + b  # PyTorch overrides the "+" operation so that it is able to act on Tensors
print(f"c1: {c1}")
print(f"c2: {c2}")

Notice how we've created a computation graph consisting of PyTorch operations, and how the output is a tensor with value 76 -- we've just created a computation graph consisting of operations, and it's executed them and given us back the result.

Now let's consider a slightly more complicated example:

![alt text](images/external/computation-graph.png)

Here, we take two inputs, `a, b`, and compute an output `e`. Each node in the graph represents an operation that takes some input, does some computation, and passes its output to another node.

Let's define a simple function in PyTorch to construct this computation function:


In [ ]:
# ── Multi-step computation graph: a,b → c,d → e ──────────────────────────────


# Construct a simple computation function
def func(a, b):
    """TODO: Define the operation for c, d, e."""
    c = torch.add(a, b)
    d = torch.subtract(b, 1)
    e = torch.multiply(c, d)
    return e

Now, we can call this function to execute the computation graph given some inputs `a,b`:


In [ ]:
# ── Execute the computation graph ─────────────────────────────────────────────
# Consider example values for a,b
a, b = 1.5, 2.5
# Execute the computation
e_out = func(a, b)
print(f"e_out: {e_out}")

#### What just happened — and what's missing

PyTorch executed our computation and returned a tensor. But it did more than arithmetic — it **recorded every operation** in a computation graph. When we later call `.backward()` on any output, PyTorch walks that graph in reverse to compute gradients for every input that has `requires_grad=True`.

**Missing piece**: We defined _what to compute_, but not _what to optimise_. For that we need a trainable model with learnable weights — that's `nn.Module`, next.


### Predict before you run

Before running the next cell, commit to an answer: using `w_size=0.15`, `w_age=-5.0`, `bias=100.0`, what will the model predict for house [0] (1200 sq ft, 10 years old, true price $250k)?

1. About **$180k** — the age penalty dominates the size contribution.
2. About **$230k** — size and age partially cancel; prediction is close but under.
3. About **$310k** — size contribution dominates and overshoots.


In [ ]:
# ── Computation on our house dataset — price prediction by hand ───────────────
size = houses[0, 0]  # 1200 sq ft
age = houses[0, 1]  # 10 years
true_price = prices[0]  # $250k

# Hand-picked weights (we'll learn better ones later via gradient descent)
w_size = torch.tensor(0.15)
w_age = torch.tensor(-5.0)
bias = torch.tensor(100.0)

contrib_size = w_size * size
contrib_age = w_age * age
price_pred = contrib_size + contrib_age + bias

print(f"House [0]: {size:.0f} sq ft, {age:.0f} years → true price ${true_price:.0f}k")
print(f"  w_size × size = {w_size:.2f} × {size:.0f} = ${contrib_size:.1f}k")
print(f"  w_age  × age  = {w_age:.2f} × {age:.0f} = ${contrib_age:.1f}k")
print(f"  + bias        = ${bias:.1f}k")
print(
    f"  → Predicted:  ${price_pred:.1f}k   (error: ${abs(price_pred - true_price):.1f}k)"
)
print(
    "\nPyTorch traced this computation automatically. We'll use that trace for autograd next."
)

### Exercise — manual weight tuning

Before Section 1.3 introduces _learnable_ weights, try tuning the three weights by hand.

**Predict**: Can you fit all 5 houses within ±$10k error just by adjusting numbers?


In [ ]:
# EXERCISE — manual weight tuning
# CHANGE these three weights to predict all 5 house prices within ±$10k error
w_size = torch.tensor(0.15)  # ← try 0.10, 0.20, 0.25...
w_age = torch.tensor(-5.0)  # ← try -3.0, -8.0...
bias = torch.tensor(100.0)  # ← try 50.0, 150.0...

predictions = houses[:, 0] * w_size + houses[:, 1] * w_age + bias
errors = torch.abs(predictions - prices)

print("Manual tuning results:")
print(f"  {'House':>6}  {'True':>8}  {'Predicted':>10}  {'Error':>8}")
for i, (p_true, p_pred, err) in enumerate(zip(prices, predictions, errors)):
    check = "✓" if err < 10.0 else "✗"
    print(f"  [{i}]     ${p_true:6.1f}k   ${p_pred:6.1f}k      ${err:5.1f}k  {check}")
mean_error = errors.mean()
print(f"\nMean absolute error: ${mean_error:.1f}k")
if mean_error < 10.0:
    print("✓ You nailed it by hand — but imagine 100 features, 10000 houses...")
else:
    print(
        "→ Hand-tuning fails even on 5 houses. We need LEARNING: autograd + gradient descent."
    )

#### What just happened — and the crack it leaves open

We manually wrote `w_size * size + w_age * age + bias` and saw the prediction fail. But:

- We **hand-picked** those weights. What if we have 100 features?
- We **guessed** they'd work. How do we find _good_ weights systematically?

That's the job of **gradient descent** — and it requires computing `∂loss/∂w_size`. That's what autograd does automatically.


---

## Part 3 — Neural Networks in PyTorch

A perceptron does one thing: take a set of inputs, multiply each by a learned weight, sum them up, then squash the result through a non-linearity so the output stays bounded. The formula shorthand for this is $y = \sigma(Wx + b)$.

The intuition behind each piece: **$W$ decides how much each feature matters** — a large positive weight for `size` means bigger houses cost more; **$b$ is a constant offset** (a baseline price even for the smallest house); **$\sigma$ (sigmoid) keeps output in $(0, 1)$** — without it, stacking multiple layers would just collapse into one giant linear function, which defeats the point. Every arrow in the diagram below is one entry in $W$ or $b$, and those numbers are exactly what the model _learns_.

![alt text](images/external/computation-graph-2.png)

PyTorch's [`torch.nn.Module`](https://pytorch.org/docs/stable/generated/torch.nn.Module.html) is the container that holds these learnable parameters and wires up the computation. You subclass it, declare weights as `nn.Parameter` objects, and override `forward()` with the actual math. PyTorch then traces every operation through those parameters so `.backward()` can compute gradients for each one automatically — no manual calculus needed.

Let's write a dense layer class to implement the perceptron above.


#### **Predict first** — what does `nn.Module` give us?

We're about to define a custom dense layer by subclassing `torch.nn.Module`. Before reading the class, predict which of these is true:

1. We must manually call `.backward()` on **each** `nn.Parameter` separately inside the class.
2. Wrapping a tensor in `nn.Parameter` is sufficient — autograd tracks it automatically from that point forward, no extra code needed.
3. We need a custom gradient-accumulation loop inside `__init__` for parameters to work.

Pick your answer, then read and run the class definition below.


In [ ]:
# ── Hand-written dense layer (nn.Module subclass) ─────────────────────────────
# num_inputs: number of input nodes
# num_outputs: number of output nodes
# x: input to the layer


class OurDenseLayer(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super(OurDenseLayer, self).__init__()
        # Define and initialize parameters: a weight matrix W and bias b
        # Note that the parameter initialize is random!
        self.W = torch.nn.Parameter(torch.randn(num_inputs, num_outputs))
        self.bias = torch.nn.Parameter(torch.randn(num_outputs))

    def forward(self, x):
        """TODO: define the operation for z (hint: use torch.matmul)."""
        z = torch.matmul(x, self.W) + self.bias

        """TODO: define the operation for out (hint: use torch.sigmoid)."""
        y = torch.sigmoid(z)
        return y


print(
    "OurDenseLayer defined — W and bias are nn.Parameter, so autograd tracks them automatically."
)
print("No extra registration needed: wrapping a tensor in nn.Parameter is sufficient.")

Now, let's test the output of our layer.


In [ ]:
# ── Test OurDenseLayer ────────────────────────────────────────────────────────
# Define a layer and test the output!
num_inputs = 2
num_outputs = 3
layer = OurDenseLayer(num_inputs, num_outputs)
x_input = torch.tensor([[1, 2.0]])
y = layer(x_input)

print(f"input shape: {x_input.shape}")
print(f"output shape: {y.shape}")
print(f"output result: {y}")

#### What just happened — and what's missing

`OurDenseLayer` ran a single forward pass: random weights `W` multiplied the 2-feature input, bias was added, sigmoid was applied, and a (1 x 3) output tensor emerged. Declaring weights as `nn.Parameter` is all that is required for PyTorch to register them for gradient tracking — no extra bookkeeping.

What's missing: every new layer type needs its own subclass. PyTorch ships pre-built blocks (`nn.Linear`, `nn.Sigmoid`) that can be composed without writing a custom `forward()` — that is `nn.Sequential`, shown next.


Conveniently, PyTorch has defined a number of `nn.Modules` (or Layers) that are commonly used in neural networks, for example a [`nn.Linear`](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html) or [`nn.Sigmoid`](https://pytorch.org/docs/stable/generated/torch.nn.Sigmoid.html) module.

Now, instead of using a single `Module` to define our simple neural network, we'll use the [`nn.Sequential`](https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html) module from PyTorch and a single [`nn.Linear` ](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html) layer to define our network. With the `Sequential` API, you can readily create neural networks by stacking together layers like building blocks.


In [ ]:
# ── Same layer via nn.Sequential (no subclassing needed) ──────────────────────

# define the number of inputs and outputs
n_input_nodes = 2
n_output_nodes = 3

# Define the model
"""TODO: Use the Sequential API to define a neural network with a
    single linear (dense!) layer, followed by non-linearity to compute z"""
model = nn.Sequential(
    nn.Linear(n_input_nodes, n_output_nodes),
    nn.Sigmoid(),
)
print(f"Sequential model: {model}")
print(
    "nn.Sequential stacks layers in order; each nn.Parameter inside is auto-registered for gradient tracking."
)

We've defined our model using the Sequential API. Now, we can test it out using an example input:


In [ ]:
# ── Test Sequential model ─────────────────────────────────────────────────────
# Test the model with example input
x_input = torch.tensor([[1, 2.0]])
model_output = model(x_input)
print(f"input shape: {x_input.shape}")
print(f"output shape: {y.shape}")
print(f"output result: {y}")

With PyTorch, we can create more flexible models by subclassing [`nn.Module`](https://pytorch.org/docs/stable/generated/torch.nn.Module.html). The `nn.Module` class allows us to group layers together flexibly to define new architectures.

As we saw earlier with `OurDenseLayer`, we can subclass `nn.Module` to create a class for our model, and then define the forward pass through the network using the `forward` function. Subclassing affords the flexibility to define custom layers, custom training loops, custom activation functions, and custom models. Let's define the same neural network model as above (i.e., Linear layer with an activation function after it), now using subclassing and using PyTorch's built in linear layer from `nn.Linear`.


In [ ]:
# ── Custom forward pass via subclassing nn.Module ─────────────────────────────


class LinearWithSigmoidActivation(nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super(LinearWithSigmoidActivation, self).__init__()
        """TODO: define a model with a single Linear layer and sigmoid activation."""
        self.linear = nn.Linear(num_inputs, num_outputs)
        self.activation = nn.Sigmoid()

    def forward(self, inputs):
        linear_output = self.linear(inputs)
        output = self.activation(linear_output)
        return output


print(
    "LinearWithSigmoidActivation defined — equivalent to nn.Sequential([nn.Linear, nn.Sigmoid]),"
)
print(
    "but subclassing gives flexibility to add branches, conditionals, or extra logic in forward()."
)

Let's test out our new model, using an example input, setting `n_input_nodes=2` and `n_output_nodes=3` as before.


In [ ]:
# ── Test LinearWithSigmoidActivation ──────────────────────────────────────────
n_input_nodes = 2
n_output_nodes = 3
model = LinearWithSigmoidActivation(n_input_nodes, n_output_nodes)
x_input = torch.tensor([[1, 2.0]])
y = model(x_input)
print(f"input shape: {x_input.shape}")
print(f"output shape: {y.shape}")
print(f"output result: {y}")

### Three ways to define the same layer — which one, when?

We just built the identical `Linear → Sigmoid` computation three different ways. They produce the
same math, but they trade off flexibility against boilerplate:

| Approach                                                         | Pros                                                                                                                                                                                          | Cons                                                                                                                                  |
| ---------------------------------------------------------------- | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------- |
| **`OurDenseLayer`** (manual `nn.Parameter`)                      | Full control over exactly what's learnable; makes the "declaring a tensor as `nn.Parameter` is all autograd needs" mechanism explicit — good for _teaching_ the concept.                      | Most boilerplate; you must re-derive `matmul + bias` for every layer type by hand.                                                    |
| **`nn.Sequential`**                                              | Fewest lines; layers stack top-to-bottom exactly as written — easiest to read for a simple feed-forward stack.                                                                                | Only works when data flows straight through in order — no branching, no conditionals, no reusing a layer's output twice.              |
| **Subclassing with `nn.Linear`** (`LinearWithSigmoidActivation`) | Combines PyTorch's tested, optimized building blocks with full control over `forward()` — branches, loops, or `if isidentity` conditionals (shown further down) are all just ordinary Python. | More typing than `Sequential` for a simple stack; the extra flexibility is wasted if the network really is a straight line of layers. |

**The pattern that matters**: production code almost never uses `OurDenseLayer`-style manual
parameters — `nn.Linear`, `nn.Conv2d`, etc. already exist and are optimized. The real choice is
`Sequential` (simple, linear stacks) vs. subclassing (anything with a conditional, branch, or
custom control flow) — further down, `LinearButSometimesIdentity` demonstrates exactly why
subclassing wins once "the data doesn't just flow straight through" stops being hypothetical.


#### What just happened — and what's missing

`nn.Module` bundles **weights + computation**: declare once, call repeatedly. PyTorch registers every `nn.Parameter` automatically — no manual bookkeeping.

**Missing piece**: The weights were _randomly initialised_ — the model outputs nonsense. We need a way to measure how wrong it is (**loss function**) and a way to systematically improve the weights (**gradient descent**). That's `autograd` — next in Part 4.


Importantly, `nn.Module` affords us a lot of flexibility to define custom models. For example, we can use boolean arguments in the `forward` function to specify different network behaviors, for example different behaviors during training and inference. Let's suppose under some instances we want our network to simply output the input, without any perturbation. We define a boolean argument `isidentity` to control this behavior:


In [ ]:
# ── Custom behavior: conditional identity pass ────────────────────────────────


class LinearButSometimesIdentity(nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super(LinearButSometimesIdentity, self).__init__()
        self.linear = nn.Linear(num_inputs, num_outputs)

    """TODO: Implement the behavior where the network outputs the input, unchanged,
        under control of the isidentity argument."""

    def forward(self, inputs, isidentity=False):
        """TODO"""
        if isidentity:
            return inputs
        return self.linear(inputs)


print(
    "LinearButSometimesIdentity defined — shows how subclassing lets you control forward-pass"
)
print(
    "behavior via ordinary Python conditionals: isidentity=True bypasses the linear layer entirely."
)

Let's test this behavior:


In [ ]:
# ── Test LinearButSometimesIdentity ───────────────────────────────────────────
# Test the IdentityModel
model = LinearButSometimesIdentity(num_inputs=2, num_outputs=3)
x_input = torch.tensor([[1, 2.0]])

"""TODO: pass the input into the model and call with and without the input identity option."""
out_with_linear = model(x_input)

out_with_identity = model(x_input, isidentity=True)

print(f"input: {x_input}")
print(
    "Network linear output: {}; network identity output: {}".format(
        out_with_linear, out_with_identity
    )
)

Now that we have learned how to define layers and models in PyTorch using both the Sequential API and subclassing `nn.Module`, we're ready to turn our attention to how to actually implement network training with backpropagation.


In [ ]:
# ── HousePriceModel: the same prediction, but with LEARNABLE weights ──────────
class HousePriceModel(torch.nn.Module):
    """Predicts house price from size and age using a single linear layer."""

    def __init__(self):
        super().__init__()
        self.w_size = torch.nn.Parameter(torch.randn(1) * 0.01)
        self.w_age = torch.nn.Parameter(torch.randn(1) * 0.01)
        self.bias = torch.nn.Parameter(torch.randn(1) * 0.01)

    def forward(self, size, age):
        return self.w_size * size + self.w_age * age + self.bias


torch.manual_seed(42)
model = HousePriceModel()
print("Model parameters (random initialization):")
for name, param in model.named_parameters():
    print(f"  {name}: {param.item():.4f}")

pred = model(houses[0, 0], houses[0, 1])
print(
    f"\nPrediction for house [0]: ${pred.item():.1f}k  (random weights → bad prediction)"
)
print("Next: we'll use autograd to LEARN better weights automatically.")

### Computation Graph and Automatic Differentiation

![Forward computation graph and reverse-mode automatic differentiation from scalar loss back to trainable parameters](images/computation-graph-and-autodiff.png)

The forward pass computes intermediate values and a scalar loss. Reverse-mode autodiff then applies the chain rule from that loss back through the graph, accumulating gradients on trainable parameters; an optimizer reads those stored gradients to update weights.


---

## Part 4 — Automatic Differentiation

Here is the core problem that makes neural networks trainable: we have a single loss number, and we need to know which direction to nudge _each_ of potentially millions of weights to make that number smaller. Computing those partial derivatives by hand — even for our 3-weight house-price model — is tedious algebra. For millions of weights it is impossible.

PyTorch solves this with [`torch.autograd`](https://pytorch.org/docs/stable/autograd.html): every time you do math on a tensor that has `requires_grad=True`, PyTorch silently records the operation in a computation graph. When you later call `.backward()` on the final loss, PyTorch walks that graph in reverse (applying the chain rule layer by layer) and deposits `∂loss/∂param` into each parameter's `.grad` attribute — all in one pass, regardless of how many parameters exist. This is [backpropagation](https://en.wikipedia.org/wiki/Backpropagation), automated.

To see the mechanism at its clearest, let's verify it on $y = x^2$ — one input, one output, derivative known analytically — before applying it to a full model:


#### **Predict first** — what does `.backward()` return?

We're about to compute the derivative of $y = x^2$ at $x = 3.0$ using PyTorch's autograd. Predict the result before running:

1. `dy_dx = 9.0` — because $y = x^2 = 9$ at $x = 3$.
2. `dy_dx = 6.0` — because $\frac{dy}{dx} = 2x$, evaluated at $x = 3$.
3. `dy_dx = 3.0` — because $x = 3$.

Only one of these is the gradient. Pick your answer, then run the cell.


In [ ]:
# ── Gradient of y = x² at x = 3 ──────────────────────────────────────────────
# requires_grad=True tells PyTorch to record all operations on x in a graph
x = torch.tensor(3.0, requires_grad=True)
y = x**2
y.backward()  # Walk the graph in reverse: dy/dx = 2x

dy_dx = x.grad
print("dy_dx of y=x^2 at x=3.0 is: ", dy_dx)
# dy/dx = 2x → at x=3: dy_dx = 6
assert dy_dx == 6.0
print("  → .backward() computed the analytic derivative automatically.")

The derivative alone does not train anything — you also need something to minimise. In neural networks that is the **loss function**: a single number measuring how wrong the current weights are. The gradient tells you the local slope of that loss surface; gradient descent says "take a small step downhill."

To build the intuition without any distractions, we will minimise $L = (x - x_f)^2$ — a parabola with a known bottom at $x_f$. We could solve it analytically (the minimum is trivially $x = x_f$), but we deliberately will _not_: instead, autograd computes $\partial L / \partial x$ at each iteration and gradient descent steps toward the bottom. This is the **exact same loop** that trains every neural network — just on one scalar instead of millions of weights.


#### A common shorthand — and where it breaks

Gradient descent is usually summarized in one line: \*"gradient descent finds **the** minimum of
the loss."\*\* Taken completely literally, that would mean: start anywhere, follow the slope
downhill, and you are mathematically guaranteed to land on the single best possible value.

That claim is about to look true further down — `L = (x - x_f)²` is a parabola with exactly
**one bowl**, so every starting point slides down to the same global minimum at `x = x_f`, no
matter where `x` starts. We picked this loss precisely because it is convex: one bowl, one
minimum, no way to get stuck.

That is _not_ the general case. A real neural network's loss surface — plotted against millions of
weights instead of one scalar `x` — is riddled with local minima, saddle points, and flat plateaus.
The exact same update rule, `x ← x − α·∇ₓL`, run on that surface is only guaranteed to reach **a**
point where the local gradient is ~zero — not necessarily the single best point in the whole space.
The shorthand isn't wrong, it's describing the special (convex) case; the toy problem below is
deliberately that special case so the mechanism is easy to see, not because real training looks
like this.


### Predict before you run

Before running the next cell, commit to an answer: when minimising `L = (x - 4)^2` starting from a random `x`, where will gradient descent converge after 500 iterations with `lr=0.01`?

1. **x converges to 0** — gradient is always negative so x keeps decreasing.
2. **x converges to 4** — the global minimum of the parabola, where the gradient is zero.
3. **x converges somewhere between the start and 4** — gradient descent never fully reaches the minimum in finite steps.


In [ ]:
# ── Gradient descent: minimize L = (x − x_f)² ────────────────────────────────
# Minimizing L = (x - x_f)^2 analytically gives x = x_f.
# Here we let gradient descent find that minimum iteratively — same mechanism
# that trains neural networks, just on a single scalar instead of millions of weights.

# Initialize a random value for our initial x
x = torch.randn(1)
print(f"Initializing x={x.item()}")

learning_rate = 1e-2  # Learning rate
history = []
x_f = 4  # Target value


# We will run gradient descent for a number of iterations. At each iteration, we compute the loss,
#   compute the derivative of the loss with respect to x, and perform the update.
for i in range(500):
    x = torch.tensor([x], requires_grad=True)

    # Compute the loss as the square of the difference between x and x_f
    loss = (x - x_f) ** 2

    # Backpropagate through the loss to compute gradients
    loss.backward()

    # Update x with gradient descent
    x = x.item() - learning_rate * x.grad

    history.append(x.item())

# Plot the evolution of x as we optimize toward x_f!
plt.plot(history)
plt.plot([0, 500], [x_f, x_f])
plt.legend(("Predicted", "True"))
plt.xlabel("Iteration")
plt.ylabel("x value")
plt.show()
print(f"  → x converged to {history[-1]:.4f}; target was {x_f}.")
print("  → Same loop — gradient × learning rate — trains every neural network.")

### Code Walkthrough: Manual Gradient Descent on a Parabola

**What just ran — 3 key patterns:**

---

**`x = torch.tensor([x], requires_grad=True)` — re-wrap each step**
Each iteration re-wraps the current scalar `x` in a **new** tensor with `requires_grad=True`. This resets the computation graph so `loss.backward()` accumulates only the gradient for this step. Without re-wrapping, PyTorch would try to differentiate through all 500 iterations at once — a memory explosion for real models.

---

**`loss.backward()` — compute `∂loss/∂x` automatically**
For `loss = (x - x_f)²`, the analytic gradient is `2(x - x_f)`. PyTorch computes this via the recorded computation graph. The result is deposited in `x.grad`. You never write the derivative by hand — this is the entire point of autograd.

---

**`x = x.item() - learning_rate * x.grad` — the SGD update rule**
`x.item()` detaches the scalar from the graph (otherwise the next iteration's tensor construction would inherit a broken graph). The update $x \leftarrow x - \alpha \cdot \nabla_x L$ steps downhill by a distance proportional to the current gradient. After 500 steps the loss is near-zero and `x` converges to `x_f = 4`.

> **PyTorch shape note:** `x.grad` always has the same shape as `x`. For a model weight tensor of shape `(128, 256)`, its `.grad` is also `(128, 256)` — one gradient entry per parameter.


In [ ]:
# ── FuncAnimation: x stepping down the parabola L = (x − x_f)² ──────────────
# Shows the same convergence as the static plot above — animated step-by-step.
# Each frame = 5 gradient descent steps; watch x glide toward x_f = 4.
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

x_f = 4
x_range = np.linspace(min(history) - 0.5, max(history) + 0.5, 300)
L_range = (x_range - x_f) ** 2

# Sample every 5th step for 100 frames
frame_step = max(1, len(history) // 100)
frame_x = history[::frame_step]
frame_L = [(x - x_f) ** 2 for x in frame_x]

fig_anim, ax_anim = plt.subplots(figsize=(9, 5))
fig_anim.patch.set_facecolor("#2d2d2d")
ax_anim.set_facecolor("#2d2d2d")
ax_anim.plot(x_range, L_range, color="white", linewidth=2, label="L = (x − 4)²")
ax_anim.axvline(
    x=x_f, color="mediumseagreen", linestyle="--", alpha=0.7, label="minimum x = 4"
)
ax_anim.legend(facecolor="#333333", labelcolor="white", loc="upper left")
ax_anim.set_xlabel("x value", color="white")
ax_anim.set_ylabel("L(x)", color="white")
ax_anim.tick_params(colors="white")
for spine in ax_anim.spines.values():
    spine.set_edgecolor("#555555")

(dot,) = ax_anim.plot([], [], "o", color="coral", markersize=12, zorder=5)
title_obj = ax_anim.set_title("", color="white")


def _update(frame):
    dot.set_data([frame_x[frame]], [frame_L[frame]])
    step = frame * frame_step
    title_obj.set_text(
        f"Step {step}/{len(history)-1}   x = {frame_x[frame]:.3f}   L = {frame_L[frame]:.4f}"
    )
    return dot, title_obj


anim = FuncAnimation(fig_anim, _update, frames=len(frame_x), interval=60, blit=False)
plt.close(fig_anim)  # prevent duplicate static frame in Jupyter

print("Animation: gradient descent stepping down the parabola L = (x − 4)².")
print("  Each frame = one sample point from the 500-step descent above.")
print("  Green dashed line = global minimum at x = 4. Watch x converge.\n")
display(HTML(anim.to_jshtml(fps=12)))

In [ ]:
# ── Visualizing what .backward() actually does ────────────────────────────────
torch.manual_seed(42)
model_viz = HousePriceModel()
optimizer = torch.optim.SGD(model_viz.parameters(), lr=1e-5)

optimizer.zero_grad()
preds = model_viz(houses[:, 0], houses[:, 1])
loss = torch.mean((preds - prices) ** 2)
loss.backward()

print(f"After loss.backward(), gradients computed:")
for name, param in model_viz.named_parameters():
    print(f"  ∂loss/∂{name}: {param.grad.item():+.4f}")

print(f"\nLoss: {loss.item():.2f}")
print("→ .backward() computed ∂loss/∂w_size, ∂loss/∂w_age, ∂loss/∂bias automatically.")
print("  Gradient descent updates each weight in the direction that reduces loss.")

# Show one update step
print(f"\nBefore update: w_size = {model_viz.w_size.item():.4f}")
optimizer.step()
print(
    f"After step:    w_size = {model_viz.w_size.item():.4f}  (moved toward lower loss)"
)

### Training Loop to Convergence

One `optimizer.step()` nudges every weight a tiny bit — barely visible in a single printout. Real
training repeats **forward → loss → backward → step** hundreds or thousands of times. Let's run
`HousePriceModel` to convergence on our 5 houses and watch the loss curve, instead of stopping
after a single step.


In [ ]:
# ── Full training loop: iterate to convergence ────────────────────────────────
# A single step barely moves the loss. Let's repeat forward → loss → backward →
# step for many epochs and watch the loss curve actually fall.
torch.manual_seed(42)
model_trained = HousePriceModel()
optimizer = torch.optim.SGD(model_trained.parameters(), lr=1e-5)

loss_history = []
for epoch in range(500):
    optimizer.zero_grad()
    preds = model_trained(houses[:, 0], houses[:, 1])
    loss = torch.mean((preds - prices) ** 2)
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())
    if epoch % 100 == 0:
        print(f"epoch {epoch:3d}  loss = {loss.item():8.2f}")
print(f"epoch {len(loss_history) - 1:3d}  loss = {loss_history[-1]:8.2f}  (final)")

plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss ($k²)")
plt.title("HousePriceModel training loop — loss vs. epoch")
plt.show()

print("\nFinal predictions vs. true prices:")
final_preds = model_trained(houses[:, 0], houses[:, 1]).detach()
for i, (p_true, p_pred) in enumerate(zip(prices, final_preds)):
    print(f"  house [{i}]  true=${p_true:.0f}k  predicted=${p_pred:.0f}k")

print(
    f"\n→ {len(loss_history)} iterations of forward → loss → backward → step took the loss from "
    f"{loss_history[0]:.1f} to {loss_history[-1]:.1f} — the exact loop that trains every neural network,"
    " just repeated many more times on many more weights."
)

#### What just happened — from toy to production

We trained a 3-parameter model with `.backward()` for 500 full iterations — not just one step — and
watched the loss curve fall toward zero. The same mechanism trains:

- GPT-2: 1.5 billion parameters
- Stable Diffusion: 860 million parameters

**The only difference is scale.** The autograd graph, the gradient computation, the optimizer step — all identical.


---

## Part 5 — From Toy to Production: Same Machinery, Bigger Numbers

Our house-price model has **3 learnable parameters**. A production deep-learning model has millions — but the _mechanism_ is identical: `nn.Parameter`, `.forward()`, `.backward()`, `optimizer.step()`.

| Model                            | Parameters   | Same autograd? | Same nn.Module? |
| -------------------------------- | ------------ | -------------- | --------------- |
| Our toy (house prices)           | 3            | ✓              | ✓               |
| ResNet-18 (image classification) | 11.7 million | ✓              | ✓               |
| GPT-2 (language model)           | 1.5 billion  | ✓              | ✓               |
| Stable Diffusion                 | 860 million  | ✓              | ✓               |

**The only difference is scale.** If you understood gradient descent on 3 weights, you understand it on 860 million.


### From Toy to Production

![Toy instructional classifier and production-scale deep-learning model share the same data-to-parameter-update learning cycle](images/toy-to-production-deep-learning.png)

Production systems add more data, capacity, compute, evaluation rigor, and operational constraints. The fundamental loop remains data, model, predictions, loss, gradients, and parameter updates; scaling does not guarantee quality by itself.


### Predict before you run

Before running the next cell, commit to an answer: our house-price model has 3 learnable parameters. Roughly how many parameters does ResNet-18 have?

1. About **11 thousand** — still small, just deeper.
2. About **11 million** — three orders of magnitude more than our toy model.
3. About **11 billion** — in the same range as large language models.


### Pretrained Music Generation Paths

![Two pretrained music-generation inference paths: text to audio and ABC notation to symbolic notation](images/pretrained-music-generation-paths.png)

The two options use different modalities: MusicGen-style systems turn a natural-language prompt into audio, while TunesFormer-style systems continue symbolic ABC notation. Both are pretrained-model inference workflows, not from-scratch training exercises.


### The Explicit Training Loop

![Comparison of high-level framework training APIs and explicit PyTorch or custom TensorFlow training loops](images/training-loop-framework-translation.png)

High-level APIs manage the loop for you, while an explicit PyTorch loop exposes the same essential operations: clear gradients, forward pass, loss, backward pass, and optimizer update. Those are different interfaces to the same learning cycle.


In [ ]:
# ── Toy-to-real: same nn.Module, different scale ──────────────────────────────
try:
    import torchvision

    real_model = torchvision.models.resnet18(weights=None)  # structure only
    total = sum(p.numel() for p in real_model.parameters())
    trainable = sum(p.numel() for p in real_model.parameters() if p.requires_grad)
    print(f"ResNet-18 architecture:")
    print(f"  Total parameters   : {total:,}")
    print(f"  Trainable parameters: {trainable:,}")
    print(f"  Layers: {len(list(real_model.children()))}")
    print(
        "\nEvery one of those 11.7M parameters is a torch.nn.Parameter, just like our w_size."
    )
    print(
        "Every forward pass traces a computation graph. Every .backward() computes all gradients."
    )
    print(
        "The machinery you learned on 3 weights scales to billions — no new concepts needed."
    )
except ImportError:
    print(
        "[torchvision not installed — the point stands: production models use the same"
    )
    print(
        " nn.Module / autograd machinery you just learned on 5 houses and 3 weights.]"
    )

#### What just happened — and what's missing

ResNet-18 has 11.7 million parameters — every one a `torch.nn.Parameter` inside an `nn.Module`, just like our `w_size` and `w_age`. The autograd graph, `.backward()` call, and optimizer step are structurally identical to what we used on 5 houses; only the count differs.

What's missing: all architectures so far accept fixed-size inputs. When inputs are variable-length sequences — sentences, melodies, time series — we need a different structure. That is recurrent networks and attention, covered in the next lab.


Parts 1–5 cover the complete PyTorch mental model — tensors, graphs, modules, autograd, and gradient descent. Part 6 applies the same autograd machinery at production scale via pre-trained music generation models.


---

## Part 6 — Music Generation with HuggingFace

Instead of training an RNN from scratch, this section lets you pick a
**pre-trained model from HuggingFace Hub** and generate music immediately.

Two models are available, selectable further down via a `MUSIC_MODEL` flag:

| Model                     | Approach            | Input                 | Output            |
| ------------------------- | ------------------- | --------------------- | ----------------- |
| `facebook/musicgen-small` | Transformer seq2seq | **text prompt**       | raw audio (WAV)   |
| `sander-wood/tunesformer` | Causal LM           | **ABC notation seed** | ABC notation text |

> **ABC notation** is a text-based music format used for Irish/Celtic folk tunes.
> Example: `X:1\nT:Title\nM:6/8\nK:Gmaj\n|: G2A B2c | d2e fed |`
> The model extends that seed, producing a complete tune you can play with `music21` or `abc2midi`.


### Which one should you actually run?

Both models generate music from a pre-trained checkpoint, but they differ enough that the right
choice depends on what you want out of the exercise:

|                                           | Pros                                                                                                                                                   | Cons                                                                                                                                                                          |
| ----------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------ | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **MusicGen** (text prompt → audio)        | You hear finished audio immediately — no extra conversion step; prompts are plain English, no notation to learn.                                       | ~300 MB download; a CPU forward pass takes noticeably longer than TunesFormer's; the output can't be hand-edited note-by-note.                                                |
| **TunesFormer** (ABC seed → ABC notation) | Small, fast download; output is plain text you can read, tweak, and re-feed as a new seed; converts cleanly to MIDI via `music21` for further editing. | You don't hear anything until you convert/play the ABC text; limited to the Irish/Celtic folk style it was fine-tuned on; requires knowing (or trusting) ABC notation syntax. |

If your goal is "hear a result fast," pick MusicGen. If your goal is "see and edit the actual
generated structure" — the closer analogue to inspecting a from-scratch RNN's output token by
token — pick TunesFormer.


### Why Pre-trained Models Instead of Training From Scratch?

You've learned PyTorch's autograd system on a 3-parameter toy model. Real-world models (GPT, BERT, MusicGen) are trained the _same way_ — just scaled up to billions of parameters and trained on massive datasets (books, web text, audio).

| Approach                  | Data Needed                    | Training Time              | Architecture         |
| ------------------------- | ------------------------------ | -------------------------- | -------------------- |
| From Scratch              | 20,000+ hours of labeled audio | Days/weeks on GPU clusters | Must design yourself |
| Pre-trained (HuggingFace) | Zero                           | Seconds (inference only)   | Already proven       |

**This section demonstrates inference** (generating music from a trained model). The _training_ used the exact same `.backward()` you just learned — just at scale.


In [ ]:
## ── Install HuggingFace dependencies ────────────────────────────────────────
## Run once; restart the kernel after installing if needed.

!pip install transformers accelerate scipy soundfile music21 --quiet


### Exercise — music generation

The next cell lets you choose between two pretrained models. For whichever you pick, change the prompt/seed and **predict** what the output will sound like before generating.


In [ ]:
## ── Model selector ──────────────────────────────────────────────────────────
## Change MUSIC_MODEL to switch between the two approaches at runtime.

# ── Pick one ─────────────────────────────────────────────────────────────────
MUSIC_MODEL = "musicgen"  # "musicgen"  |  "tunesformer"

# ── MusicGen config (used when MUSIC_MODEL == "musicgen") ────────────────────
#   Model sizes: musicgen-small (~300 MB) | musicgen-medium (~1.5 GB) | musicgen-large (~3.3 GB)
MUSICGEN_REPO = "facebook/musicgen-small"
MUSICGEN_PROMPT = "upbeat Irish folk music with fiddle and flute, lively jig"
MUSICGEN_DURATION = 8  # seconds of audio to generate

# ── TunesFormer config (used when MUSIC_MODEL == "tunesformer") ──────────────
#   TunesFormer generates ABC notation conditioned on a control-code seed.
#   Seed format:  X:<index>  T:<title>  M:<time sig>  K:<key>  then barlines.
TUNESFORMER_REPO = "sander-wood/tunesformer"
TUNESFORMER_SEED = "X:1\nT:My Generated Tune\nM:6/8\nL:1/8\nK:Gmaj\n|: G2A B2c |"
TUNESFORMER_NEW_TOKENS = 400  # max new tokens (≈ 1-2 full tunes)
TUNESFORMER_TEMPERATURE = 0.9
TUNESFORMER_TOP_K = 50

OUTPUT_WAV = "generated_music.wav"

print(f"Selected model: {MUSIC_MODEL!r}")

### Option A — `facebook/musicgen-small` (text-prompt → audio)

MusicGen is a Transformer encoder-decoder trained by Meta AI on 20 000 hours of
licensed music. You describe the music you want in plain English and it generates
a raw audio waveform directly — no ABC notation, no MIDI intermediate step.

The cells further down run when `MUSIC_MODEL = "musicgen"`: load the pipeline, generate audio,
then save and play it.


In [ ]:
if MUSIC_MODEL == "musicgen":
    from transformers import pipeline
    import scipy.io.wavfile
    import numpy as np
    import IPython.display as ipd

    print(f"Loading {MUSICGEN_REPO} ...")
    musicgen_pipe = pipeline(
        "text-to-audio",
        model=MUSICGEN_REPO,
        device="cpu",  # change to 0 (or "cuda") if a GPU is available
    )

With the pipeline loaded, generate audio from the text prompt:


In [ ]:
if MUSIC_MODEL == "musicgen":
    print(
        f"Generating {MUSICGEN_DURATION}s of audio for prompt:\n  '{MUSICGEN_PROMPT}'"
    )
    result = musicgen_pipe(
        MUSICGEN_PROMPT,
        forward_params={
            "do_sample": True,
            "max_new_tokens": int(MUSICGEN_DURATION * 50),
        },
    )

Normalise the raw float audio to a WAV-compatible `int16` range, save it to disk, and play it
inline:


In [ ]:
if MUSIC_MODEL == "musicgen":
    audio = result["audio"].squeeze()
    sr = result["sampling_rate"]

    # Normalise to int16 for WAV export
    audio_int16 = (audio / np.abs(audio).max() * 32767).astype(np.int16)
    scipy.io.wavfile.write(OUTPUT_WAV, sr, audio_int16)
    print(f"Saved to {OUTPUT_WAV}")

    ipd.display(ipd.Audio(audio, rate=sr))

### Code Walkthrough: MusicGen — Text-to-Audio via HuggingFace Pipeline

**What just ran — 3 key patterns:**

---

**`pipeline("text-to-audio", model=MUSICGEN_REPO)` — one-line model loading**
`pipeline` downloads the model weights, tokenizer, and processor from the HuggingFace Hub and wraps them in a ready-to-call object. Under the hood this is an `AutoModel.from_pretrained` call plus a task-specific pre/post-processing wrapper — the same `nn.Module` machinery you just learned, packaged for convenience.

---

**`musicgen_pipe(MUSICGEN_PROMPT, forward_params={...})` — conditional generation**
The pipeline tokenizes the text prompt, passes it through MusicGen's encoder, and autoregressively generates audio tokens. `do_sample=True` enables stochastic sampling (the model samples from the probability distribution at each step rather than always picking the most likely token). `max_new_tokens` caps the output length in units of 20 ms audio frames — `duration × 50` converts seconds to frames.

---

**`(audio / np.abs(audio).max() * 32767).astype(np.int16)` — float → WAV normalisation**
The raw output is a float32 array in `[-1, 1]`. Multiplying by 32767 scales it to the int16 range before writing a WAV file. Without normalisation, quiet audio or a `scipy.io.wavfile` type error would result.


### Option B — `sander-wood/tunesformer` (ABC seed → ABC notation)

TunesFormer is a GPT-style causal LM fine-tuned on thousands of Irish/Celtic folk
tunes in ABC notation. You supply a short **seed** (title, metre, key, a bar or
two) and the model completes the tune character-by-character — the same mechanism
as the MIT from-scratch LSTM, but using a pretrained model.

The output is **ABC notation text** which you can:

- Copy into [https://abc.rectanglered.com](https://abc.rectanglered.com) to hear it
- Convert to MIDI with `music21` (shown further down)
- Convert to audio with `timidity` or GarageBand

The cells further down run when `MUSIC_MODEL = "tunesformer"`: load the model, define the
generation helper, run it, then optionally convert the result to MIDI.


In [ ]:
if MUSIC_MODEL == "tunesformer":
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM

    print(f"Loading {TUNESFORMER_REPO} ...")
    tf_tokenizer = AutoTokenizer.from_pretrained(TUNESFORMER_REPO)
    lm_model = AutoModelForCausalLM.from_pretrained(TUNESFORMER_REPO)
    lm_model.eval()

With the model loaded, define a completion-only generation helper — it strips the seed from the
output so the returned string picks up exactly where the seed ended (the same pattern used by the
`generate()` helper in `01-llm-finetuning-data-techniques.ipynb`):


In [ ]:
if MUSIC_MODEL == "tunesformer":

    def generate_abc(
        model,
        tokenizer,
        seed: str,
        max_new_tokens: int = 400,
        temperature: float = 0.9,
        top_k: int = 50,
    ) -> str:
        """Generate ABC notation continuing from `seed`.

        Returns ONLY the newly generated tokens (not the prompt), so the
        returned string picks up exactly where the seed ended.
        Falls back to a descriptive message when the model emits no new content.

        Args:
            model: A causal-LM loaded with AutoModelForCausalLM.
            tokenizer: Matching tokenizer for `model`.
            seed: ABC seed string (header lines + first bar or two).
            max_new_tokens: Maximum number of new tokens to generate.
            temperature: Sampling temperature; higher = more varied output.
            top_k: Restrict sampling to the k most probable next tokens.

        Returns:
            Newly generated ABC text (prompt stripped), or a fallback string.
        """
        enc = tokenizer(seed, return_tensors="pt")
        prompt_len = enc["input_ids"].shape[1]  # number of prompt tokens to strip later
        with torch.no_grad():
            out_ids = model.generate(
                enc["input_ids"],
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                top_k=top_k,
                pad_token_id=tokenizer.eos_token_id,
            )
        new_tokens = out_ids[0][prompt_len:]  # strip prompt — only the completion
        completion = tokenizer.decode(new_tokens, skip_special_tokens=True)
        if not completion.strip():
            return (
                "[No new tokens generated — try a larger TUNESFORMER_NEW_TOKENS value]"
            )
        return completion

Run the helper on the configured seed and print the completion:


In [ ]:
if MUSIC_MODEL == "tunesformer":
    print("Generating ABC notation ...")
    generated_abc = generate_abc(
        lm_model,
        tf_tokenizer,
        TUNESFORMER_SEED,
        max_new_tokens=TUNESFORMER_NEW_TOKENS,
        temperature=TUNESFORMER_TEMPERATURE,
        top_k=TUNESFORMER_TOP_K,
    )
    print("\n-- Generated ABC notation (new tokens only) ------------------------")
    print(generated_abc)
    print("--------------------------------------------------------------------")

Optionally convert the seed + completion into a playable MIDI file:


In [ ]:
if MUSIC_MODEL == "tunesformer":
    # Optional: convert to MIDI with music21
    try:
        from music21 import converter, midi

        # Combine seed + completion for a valid ABC tune with headers
        full_abc = TUNESFORMER_SEED + generated_abc
        score = converter.parse(full_abc, format="abc")
        mf = midi.translate.music21ObjectToMidiFile(score)
        midi_path = "generated_tune.mid"
        mf.open(midi_path, "wb")
        mf.write()
        mf.close()
        print(f"MIDI saved to {midi_path}")
    except Exception as e:
        print(f"[music21 MIDI export skipped: {e}]")
        print(
            "Tip: paste the ABC text above into https://abc.rectanglered.com to hear it."
        )

#### What just happened — and what's missing

Whether you ran MusicGen or TunesFormer, inference used a pre-trained model whose weights were learned via the exact `.backward()` and `optimizer.step()` loop from Part 4 — just scaled to millions of parameters across weeks of GPU training. The audio waveform or ABC notation text is the output of a single `model.forward()` call.

What's missing: this notebook covered PyTorch fundamentals end-to-end. The next lab applies the same machinery to sequence modelling — RNNs, LSTMs, and attention — where the running example is text or audio rather than a 5-row table.


---

## What This Notebook Covered (and What It Didn't)

"PyTorch fundamentals" spans a wide topic space — tensor mechanics, autograd internals, model-building
APIs, full training pipelines, and production-scale deployment could each fill their own notebook.
Here is every technique from that space sorted into exactly one tier, so nothing is left as a bare
name with no verdict attached.

### Tier 1 — Implemented and Demonstrated

**Tensors:**

- Tensor creation & properties — scalars → vectors → matrices → 4-D batches, `.shape`, `.ndim`
- Indexing/slicing — row, column, and scalar extraction from a matrix
- NumPy ↔ PyTorch interop — `torch.from_numpy` (zero-copy), measured speed comparison
- Shape-mismatch safety — a real `RuntimeError` raised on incompatible shapes
- GPU availability check — `torch.cuda.is_available()`
- Batch processing — all 5 houses processed in one forward call (implicit vectorization)

**Computation graphs & autograd:**

- Building a computation graph from ops (`torch.add`, `+`, the multi-step `func(a, b)` example)
- `requires_grad`, `.backward()`, and reading `.grad` off a tensor (`y = x²` example)
- Manual gradient descent on a scalar loss (`L = (x - x_f)²`), iterated to convergence
- `torch.optim.SGD` — both a single step and, after this pass, a full training loop

**Model building:**

- `nn.Parameter` + hand-written `forward()` (`OurDenseLayer`)
- `nn.Sequential`
- Subclassing `nn.Module` with custom `forward()`, including conditional control flow (`LinearButSometimesIdentity`)

**Training loop (promoted this pass — see Part 4):**

- `HousePriceModel` trained to convergence over 500 epochs, with a real loss-vs-epoch curve and a
  final-vs-true price comparison — not just the single `optimizer.step()` shown before this pass

**Scaling to production:**

- Real parameter counts for a real architecture (`torchvision.models.resnet18`, `.numel()`)
- Real pretrained inference — HuggingFace `pipeline()` (MusicGen) and `AutoModelForCausalLM.generate()` (TunesFormer)
- Real decoding controls — `temperature`, `top_k` in `generate_abc()`
- Reproducibility — `torch.manual_seed`, `np.random.seed`

### Tier 2 — Explained but Not Fully Implemented

- **GPU tensor transfer** (`.to('cuda')`) — availability is checked and the API is named, but no
  tensor is actually moved because this environment has no GPU
- **Non-convex loss surfaces / local minima** — the "common shorthand — and where it breaks" section
  explains, in prose, why `L = (x - x_f)²`'s single bowl is a convex special case and why a real
  network's loss surface isn't guaranteed a single global minimum, but no non-convex surface is
  actually built or visualised

### Tier 3 — Named but Out of Scope

- **Tensor manipulation** — `reshape`/`view`/`squeeze`/`unsqueeze`/`permute`, fancy/boolean indexing,
  in-place ops (`add_`) and their autograd caveats, dtype casting — this notebook's toy dataset never
  needs them; they belong in a dedicated tensor-manipulation lesson
- **Broadcasting** — named once, in passing, before Part 1 even starts ("no structure that can batch,
  broadcast, or GPU-accelerate it") but never explained or demonstrated as its own rule, even though
  it's used silently throughout (e.g. `w_size * size`) — flagged here rather than left as an
  unexamined assumption
- **Advanced autograd** — `torch.autograd.Function` (custom autograd), higher-order gradients
  (`create_graph=True`), vector-Jacobian products for non-scalar outputs, gradient clipping — beyond
  what's needed to build first-principles intuition
- **Mini-batch training** — `Dataset`/`DataLoader`, shuffling, train/validation splits, regularization
  (dropout, weight decay, batch norm, early stopping) — the 5-row toy dataset fits in one batch and
  isn't at risk of overfitting in a way worth demonstrating here
- **Alternative optimizers & schedules** — Adam, RMSprop, momentum, weight decay, learning-rate
  schedules — plain SGD is enough to show the mechanism; `01-llm-finetuning-data-techniques.ipynb` uses Adam at
  production scale
- **Model checkpointing** — `state_dict()`, `torch.save`/`load_state_dict` — no long-running job here
  needs resuming
- **Weight initialization strategies** — Xavier/Kaiming/default schemes — a small-random init
  (`torch.randn(1) * 0.01`) is used implicitly without discussing the general principle or alternatives
- **Classification losses** — cross-entropy/NLL — house-price prediction is a regression problem, so
  MSE is the natural fit
- **Vision/sequence-specific layers** — `nn.Conv2d`, `nn.LSTM`/`nn.RNN`, `nn.Embedding` — named only;
  building actual layers of these types belongs to a computer-vision or sequence-modeling notebook
- **Decoding strategies beyond temperature/top-k** — nucleus/top-p sampling, beam search, greedy
  decoding — temperature + top-k already demonstrate the sampling concept
- **Performance & deployment** — mixed precision, `torch.compile`, TorchScript/ONNX export — out of
  scope for a first-principles notebook
- **Sequence modeling** — vanilla RNN/LSTM/GRU mechanics, vanishing/exploding gradients in deep
  unrolled nets, attention — explicitly deferred to **Lab 2**, as this notebook's own Part 5/6 closing
  cells already say

No technique named anywhere in this notebook — including this recap — is left without a tier.


---

## Summary — What You Built

| Step | Concept                       | Key Idea                                                          | ✓   |
| ---- | ----------------------------- | ----------------------------------------------------------------- | --- |
| 1    | Tensors as Data Containers    | Scalars → vectors → matrices → batches; faster + safer than lists | ✓   |
| 2    | Operations on Tensors         | PyTorch builds computation graphs automatically                   | ✓   |
| 3    | The Manual Prediction Problem | Hand-tuning 3 weights fails; 100 features is impossible           | ✓   |
| 4    | Neural Networks in PyTorch    | nn.Module wraps learnable nn.Parameters                           | ✓   |
| 5    | Automatic Differentiation     | .backward() computes ∂loss/∂every_weight in one call              | ✓   |
| 6    | Gradient Descent in Action    | Iterative weight updates drive loss toward zero                   | ✓   |
| 7    | From Toy to Production        | Same autograd scales from 3 params to 1.5 billion (GPT-2)         | ✓   |
| 8    | Music Generation Bonus        | Pre-trained transformers (MusicGen, TunesFormer)                  | ✓   |

### Key Insights to Keep

- **Tensors are purpose-built**: GPU-ready, shape-safe, and 5-50× faster than lists for matrix operations.
- **nn.Module = learnable weights + forward pass**: The same pattern scales from 3 parameters to 1.5 billion.
- **Autograd is automatic calculus**: `.backward()` computes every ∂loss/∂param in one call, no manual derivatives needed.
- **Gradient descent is iterative refinement**: Each step nudges weights to reduce loss — cumulative tiny improvements converge to near-optimal.
- **Pre-trained models save months**: Training GPT-2 from scratch costs $50K+ in compute; inference on a pre-trained model takes seconds.
- **From toy to production, the machinery is identical**: If you understood gradient descent on house prices (3 weights), you understand it on Stable Diffusion (860M weights). Scale is the only difference.

**Next**: Dive into recurrent networks (Lab 2) and convolutional networks (Lab 3) — both built on the same PyTorch primitives you just mastered.


---

## When to Use What — PyTorch Patterns from This Notebook

| Situation                                              | Pattern to reach for                                                      | Why                                                                 |
| ------------------------------------------------------ | ------------------------------------------------------------------------- | ------------------------------------------------------------------- |
| Building a custom layer with learnable weights         | `nn.Module` subclass with `nn.Parameter`                                  | Registers weights for autodiff and `optimizer.step()` automatically |
| Simple stack of standard layers                        | `nn.Sequential`                                                           | Less boilerplate than subclassing; no custom `forward()` needed     |
| Layer with conditional logic (skip, identity path)     | `nn.Module` subclass with explicit `forward()`                            | `nn.Sequential` can't branch                                        |
| Computing gradients of a custom loss                   | `loss.backward()` + `optimizer.step()`                                    | PyTorch's dynamic graph traces the op automatically                 |
| Verifying a training loop doesn't accumulate gradients | Check gradient before `zero_grad()`; run the `zero_grad ablation` pattern | The omission is silent — no error, just wrong gradients             |
| Monitoring convergence during training                 | Append `loss.item()` to a list each epoch; plot after                     | `model.fit()` history equivalent in the manual loop                 |
| Transferring a pretrained model to inference-only      | `model.eval()` + `torch.no_grad()` context manager                        | Disables dropout; skips gradient storage (saves ~30% memory)        |
| Running the same code on CPU and GPU                   | `.to(device)` for both `model` and every batch tensor                     | Never hardcode `"cuda"` or `"cpu"` — let `device` decide            |


## Sequence Tensor Contract

This notebook uses tabular tensors such as `(batch, features)`. Sequence models add an ordered axis: `(batch, time, features)`. For tokenized language, `features` is often an embedding dimension, so a batch is a sequence of token vectors rather than independent rows.

Keep this contract in view: a model may return one prediction for each sequence, `(batch, output_features)`, or one prediction per time step, `(batch, time, output_features)`. Padding and masks identify which time positions are real when examples have different lengths.


## Next Module: Sequence Models to Transformers

This is a PyTorch and autograd foundations notebook, not an RNN-mechanics notebook. It does not build recurrent state updates, backpropagation through time, LSTM/GRU gates, or attention.

Next, use the sequence tensor contract to build a recurrent model that carries state across time. Then compare its sequential memory path with transformer self-attention, where each token can weigh other valid token positions directly. The same `nn.Module`, loss, autograd, and optimizer pattern from this notebook remains in place; the model's data flow is what changes.
